In [ ]:
import itertools
import time

import h5py
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

In [ ]:
import stretch_body.robot

# ref: https://docs.hello-robot.com/0.3/python/moving/

robot = stretch_body.robot.Robot()

did_startup = robot.startup()
print(f"Robot connected to hardware: {did_startup}")

is_homed = robot.is_homed()
print(f"Robot is homed: {is_homed}")

In [ ]:
%pip install pykalman

# Presets

- stepper: slow / default / fast / max / trajectory_max
- dynamixel: default / max / trajectory_max

```python
r.arm.move_to(
    0.3,
    v_m=r.arm.params['motion']['max']['vel_m'],
    a_m=r.arm.params['motion']['max']['accel_m']
)

r.lift.move_to(
    0.6,
    v_m=r.lift.params['motion']['slow']['vel_m'],
    a_m=r.lift.params['motion']['slow']['accel_m']
)

r.base.translate_by(
    0.5,
    v_m=r.base.params['motion']['max']['vel_m'],
    a_m=r.base.params['motion']['max']['accel_m']
)

r.end_of_arm.move_to(
    'wrist_yaw',
    1.57,
    v_r=r.end_of_arm.get_joint('wrist_yaw').params['motion']['max']['vel'],
    a_r=r.end_of_arm.get_joint('wrist_yaw').params['motion']['max']['accel']
)
```

# Motion Limits

- hard
- collision
- user
- current

```python
print(r.arm.soft_motion_limits['hard'])   # [0.0, 0.52]  (meters)
print(r.lift.soft_motion_limits['hard'])  # [0.0, 1.1]   (meters)

print(r.end_of_arm.get_joint('wrist_yaw').soft_motion_limits['hard'])    # (radians)
print(r.end_of_arm.get_joint('wrist_pitch').soft_motion_limits['hard'])  # (radians)
print(r.end_of_arm.get_joint('wrist_roll').soft_motion_limits['hard'])   # (radians)
print(r.head.get_joint('head_pan').soft_motion_limits['hard'])           # (radians)
print(r.head.get_joint('head_tilt').soft_motion_limits['hard'])          # (radians)
```

# Data Collection

都以 15 hz 蒐集 (dt = 1/15)

針對 lift, arm 的 5 種 preset 做運動, wrist yaw/pitch/roll, head pan/tilt 的 3 種 preset 做運動

利用 hard motion limit 分成四等分 (得到五個點, 假設是 0, 0.25, 0.5, 0.75, 1)

每次順序是 0 -> 1 -> 0.25 -> 0.75 -> 0.5 (來回) => 四段

每一段直接等 30 seconds 等待動作完成 (總共會是 5  * 4 段 for stepper, 3 * 4 段 for dynamixel)

每次都要存下每個 dt 的 status (pos + vel)

```
calibration_data.h5
├── arm/
│   ├── slow/
│   │   ├── seg_0/   (0.0 → 0.52)
│   │   │   ├── pos  [N,]  float32
│   │   │   └── vel  [N,]  float32
│   │   ├── seg_1/   (0.52 → 0.13)
│   │   ├── seg_2/   (0.13 → 0.39)
│   │   └── seg_3/   (0.39 → 0.26)
│   ├── default/
│   ├── fast/
│   ├── max/
│   └── trajectory_max/
└── lift/
    ├── slow/
    │   ├── seg_0 ...
    ...
```

In [ ]:
STEPPER_JOINTS = ["arm", "lift"]
STEPPER_PRESETS = ["slow", "default", "fast", "max", "trajectory_max"]

DXL_JOINTS = ["wrist_yaw", "wrist_pitch", "wrist_roll", "head_pan", "head_tilt"]
DXL_PRESETS = ["default", "max", "trajectory_max"]

In [ ]:
# Settings
DT = 1 / 15.0
SEG_DURATION = 30.0  # seconds
SETTLE_TIME = 15.0  # seconds


def get_joint_obj(name):
    if name in ("arm", "lift"):
        return getattr(robot, name)
    elif name in ("wrist_yaw", "wrist_pitch", "wrist_roll"):
        return robot.end_of_arm.get_joint(name)
    else:  # head_pan, head_tilt
        return robot.head.get_joint(name)


def get_waypoints(joint):
    lo, hi = joint.soft_motion_limits["hard"]
    pts = [lo + i * (hi - lo) / 4 for i in range(5)]  # 等分 4 段 (5 點)
    return [pts[0], pts[4], pts[1], pts[3], pts[2]]  # 0 → 1 → 0.25 → 0.75 → 0.5


def move_to(name, joint, pos, v, a):
    if name in ("arm", "lift"):
        joint.move_to(pos, v_m=v, a_m=a)
        robot.push_command()
    elif name in ("wrist_yaw", "wrist_pitch", "wrist_roll"):
        robot.end_of_arm.move_to(name, pos, v_r=v, a_r=a)
    else:  # head_pan, head_tilt
        robot.head.move_to(name, pos, v_r=v, a_r=a)


def collect_segment(name, joint, target, v, a):
    """
    Move to target and collect pos/vel for SEG_DURATION seconds.
    """
    move_to(name, joint, target, v, a)
    pos_buf, vel_buf = [], []
    t_end = time.time() + SEG_DURATION
    while time.time() < t_end:
        t0 = time.time()
        robot.pull_status()
        pos_buf.append(joint.status["pos"])
        vel_buf.append(joint.status["vel"])
        time.sleep(max(0, DT - (time.time() - t0)))
    return np.array(pos_buf, dtype=np.float32), np.array(vel_buf, dtype=np.float32)


with h5py.File("kf_data.h5", "w") as f:
    all_joints = [(name, STEPPER_PRESETS) for name in STEPPER_JOINTS] + [
        (name, DXL_PRESETS) for name in DXL_JOINTS
    ]

    for joint_name, presets in all_joints:
        joint = get_joint_obj(joint_name)
        waypoints = get_waypoints(joint)
        segments = list(itertools.pairwise(waypoints))  # 4 段

        for preset in presets:
            key = "vel_m" if joint_name in STEPPER_JOINTS else "vel"
            akey = "accel_m" if joint_name in STEPPER_JOINTS else "accel"
            v = joint.params["motion"][preset][key]
            a = joint.params["motion"][preset][akey]
            print(f"\n[{joint_name}/{preset}]  v={v:.3f}  a={a:.3f}")

            for seg_idx, (start, end) in enumerate(segments):
                # 先移動到起點等待穩定
                move_to(joint_name, joint, start, v, a)
                time.sleep(SETTLE_TIME)

                print(f"  seg {seg_idx}: {start:.3f} → {end:.3f}", end=" ... ")
                pos, vel = collect_segment(joint_name, joint, end, v, a)

                grp = f.require_group(f"{joint_name}/{preset}/seg_{seg_idx}")
                grp.create_dataset("pos", data=pos, compression="gzip")
                grp.create_dataset("vel", data=vel, compression="gzip")
                grp.attrs["start"] = start
                grp.attrs["end"] = end
                grp.attrs["preset"] = preset
                print(f"saved {len(pos)} samples")

print("\nDone! Data saved to kf_data.h5")

In [ ]:
robot.stop()

In [ ]:
def get_presets(joint_name):
    return STEPPER_PRESETS if joint_name in STEPPER_JOINTS else DXL_PRESETS


def plot_segment(joint_name, preset, seg_idx):
    with h5py.File("kf_data.h5", "r") as f:
        grp = f[f"{joint_name}/{preset}/seg_{seg_idx}"]
        pos = grp["pos"][:]
        vel = grp["vel"][:]
        start = grp.attrs["start"]
        end = grp.attrs["end"]

    t = np.arange(len(pos)) / 15.0
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=["Position", "Velocity"])
    fig.add_trace(go.Scatter(x=t, y=pos, mode="lines", name="pos"), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=vel, mode="lines", name="vel"), row=2, col=1)
    unit = "m" if joint_name in STEPPER_JOINTS else "rad"
    fig.update_layout(
        title=f"{joint_name} / {preset} / seg_{seg_idx}  ({start:.3f} → {end:.3f} {unit})",
        xaxis2_title="Time (s)",
        yaxis_title=unit,
        yaxis2_title=f"{unit}/s",
        height=500,
        showlegend=False,
    )
    return fig


# idgets
joint_dd = widgets.Dropdown(options=STEPPER_JOINTS + DXL_JOINTS, description="Joint:")
preset_dd = widgets.Dropdown(options=get_presets(joint_dd.value), description="Preset:")
seg_dd = widgets.Dropdown(options=[0, 1, 2, 3], description="Segment:")
output = widgets.Output()


def update_presets(change):
    preset_dd.options = get_presets(change["new"])


def refresh(_=None):
    output.clear_output(wait=True)
    with output:
        plot_segment(joint_dd.value, preset_dd.value, seg_dd.value).show()


joint_dd.observe(update_presets, names="value")
for w in (joint_dd, preset_dd, seg_dd):
    w.observe(refresh, names="value")

display(widgets.HBox([joint_dd, preset_dd, seg_dd]), output)
refresh()